# N4 — Fusion des deux sources et imputation Monte Carlo

Ce notebook réunit la base d'enquête nettoyée (N1) et le questionnaire nettoyé (N2) en une base unique :

1. **diagnostic** des colonnes communes et spécifiques à chaque source (opérations ensemblistes) ;
2. **harmonisation** des noms et des encodages ;
3. **concaténation verticale** (`pd.concat`), qui crée des valeurs manquantes là où une source n'a pas la colonne de l'autre ;
4. **imputation Monte Carlo** de ces manquants par tirage aléatoire dans la distribution observée.

> ⚠️ **Données synthétiques** — ce notebook s'exécute sur des données générées par `generate_dummy_data.py`. Les valeurs, proportions et graphiques affichés sont aléatoires et ne décrivent aucune population réelle.

| | Fichiers |
|---|---|
| **Entrée** | `immigration_nettoyee.xlsx` (N1) et `base_questionnaire_nettoyee.xlsx` (N2) |
| **Sortie** | `data/3_Donnees_fusionnees/base_concatenee_finale.xlsx` |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df2 = pd.read_excel("../data/2_Donnees_nettoyees/immigration_nettoyee.xlsx")

df_quest = pd.read_excel("../data/2_Donnees_nettoyees/base_questionnaire_nettoyee.xlsx")

print(f"Base immigration : {df2.shape[0]} lignes × {df2.shape[1]} colonnes")
print(f"Base questionnaire : {df_quest.shape[0]} lignes × {df_quest.shape[1]} colonnes")

Base immigration : 5000 lignes × 21 colonnes
Base questionnaire : 62 lignes × 43 colonnes


## Diagnostic des colonnes communes et spécifiques

`pd.concat` aligne les colonnes sur leur nom exact : deux variables identiques nommées différemment produiraient deux colonnes distinctes. Les intersections et différences d'ensembles (`&`, `-`) sur les noms de colonnes classent chaque variable en trois groupes : commune, propre à la base d'enquête, propre au questionnaire.

In [2]:
colonnes_immigration = set(df2.columns)
colonnes_quest = set(df_quest.columns)

communes = colonnes_immigration & colonnes_quest
specifiques_immigration = colonnes_immigration - colonnes_quest
specifiques_quest = colonnes_quest - colonnes_immigration

print(f"Colonnes communes ({len(communes)}) :")
for com in (sorted(communes)) :
  print(com)

print(f"\nColonnes uniquement dans la base immigration ({len(specifiques_immigration)}) :")
for imm in (sorted(specifiques_immigration)) :
  print(imm)

print(f"\nColonnes uniquement dans le questionnaire ({len(specifiques_quest)}) :")
for quest in (sorted(specifiques_quest)) :
  print(quest)

Colonnes communes (17) :
Age
Bankaccount
Corruption (0  1)
Country
Educationf
Genderf
GeneralizedTrsut
Professional situationf
Regionf
Remit
Social statusf
TrustCourts
TrustLocalGovernement
Trust_Parliament
Trust_Regiol_Governement
intention_to_migrate_illegal
intention_to_migrate_legal

Colonnes uniquement dans la base immigration (4) :
COUNTRY_ID
Corruption
DATE
ID

Colonnes uniquement dans le questionnaire (26) :
Acces_credit
COUNRY_ID
Choc_economique
Decideur_financier
Frequence_envois
Frequence_virements
Intention_envoyer_argent
Mariage_blanc
Montant_envois
Nationalite_francaise
Pays_etape
Proche_ami
Proche_autre
Proche_fratrie
Proche_parent
Proches_europe
Raisons_migration
Retour_pays
Revenu
Source_revenu_agriculture
Source_revenu_aide_familiale
Source_revenu_autre
Source_revenu_commerce
Source_revenu_salariat
Source_revenu_transferts
Taille_menage


## Harmonisation des colonnes avant concaténation

- correction d'une faute de frappe dans le questionnaire (`COUNRY_ID` → `COUNTRY_ID`) pour que la colonne soit reconnue comme commune ;
- suppression des colonnes administratives de la base d'enquête (`ID`, `DATE`), inutiles à l'analyse et qui resteraient vides pour les lignes du questionnaire ;
- vérification de l'encodage des colonnes communes (bloc suivant).

In [3]:
df_quest = df_quest.rename(columns={"COUNRY_ID": "COUNTRY_ID"})

print("Renommage effectué")
print(f"COUNTRY_ID maintenant dans df_quest : {'COUNTRY_ID' in df_quest.columns}")

Renommage effectué
COUNTRY_ID maintenant dans df_quest : True


In [4]:
colonnes_a_supprimer = ["ID", "DATE"]

df2 = df2.drop(columns=colonnes_a_supprimer)

print(f"Colonnes supprimées : {colonnes_a_supprimer}")
print(f"Base immigration : {df2.shape[1]} colonnes restantes")

Colonnes supprimées : ['ID', 'DATE']
Base immigration : 19 colonnes restantes


## Vérification des encodages des colonnes communes

Pour chaque colonne commune, le bloc affiche les premières valeurs uniques triées de chaque source. Une variable codée 0/1 d'un côté et 1/2 de l'autre rendrait la fusion incohérente et imposerait un recodage préalable.

In [5]:
colonnes_communes = ["Age", "Bankaccount", "Corruption (0  1)", "Country",
                     "Educationf", "Genderf", "GeneralizedTrsut",
                     "Professional situationf", "Regionf", "Remit",
                     "Social statusf", "TrustCourts", "TrustLocalGovernement",
                     "Trust_Parliament", "Trust_Regiol_Governement",
                     "intention_to_migrate_illegal", "intention_to_migrate_legal",
                     "COUNTRY_ID"]

for col in colonnes_communes:
    if col in df2.columns and col in df_quest.columns:
        valeurs_immigration = sorted(df2[col].dropna().unique())
        valeurs_quest = sorted(df_quest[col].dropna().unique())
        print(f"\n{col} :")
        print(f"  Immigration : {valeurs_immigration[:8]}{'...' if len(valeurs_immigration) > 8 else ''}")
        print(f"  Questionnaire : {valeurs_quest[:8]}{'...' if len(valeurs_quest) > 8 else ''}")


Age :
  Immigration : [np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]...
  Questionnaire : [np.int64(19), np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27)]...

Bankaccount :
  Immigration : [np.int64(0), np.int64(1)]
  Questionnaire : [np.int64(0), np.int64(1)]

Corruption (0  1) :
  Immigration : [np.int64(0), np.int64(1)]
  Questionnaire : [np.int64(0), np.int64(1)]

Country :
  Immigration : ['ALGERIA', 'EGYPT', 'IRAQ', 'JORDAN', 'LEBANON', 'LIBYA', 'MAURITANIA', 'MOROCCO']...
  Questionnaire : ['ALBANIA', 'ALGERIA', 'BENIN', 'BURKINA FASO', 'ECUADOR', 'ITALY', 'IVORY COAST', 'KOSOVO']...

Educationf :
  Immigration : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
  Questionnaire : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Genderf :
  Immigration : [np.int64(0), np.int64(1)]
 

## Correction des codes pays hors nomenclature

La plateforme de sondage attribue aux pays absents de la nomenclature de la base d'enquête des codes `COUNTRY_ID` hors plage (format 99xxx, supérieurs au plus grand code de la nomenclature). Ces codes ne correspondent à aucune modalité commune : ils sont remplacés par `NaN` par indexation booléenne (`df.loc[masque, col] = np.nan`). Le libellé texte `Country` reste disponible pour identifier ces pays.

In [6]:
nb_avant = (df_quest["COUNTRY_ID"] > 22).sum()

df_quest.loc[df_quest["COUNTRY_ID"] > 22, "COUNTRY_ID"] = np.nan

print(f"Nombre de COUNTRY_ID > 22 mis à NaN : {nb_avant}")
print(f"Valeurs uniques de COUNTRY_ID dans le questionnaire après correction :")
print(sorted(df_quest["COUNTRY_ID"].dropna().unique()))

Nombre de COUNTRY_ID > 22 mis à NaN : 47
Valeurs uniques de COUNTRY_ID dans le questionnaire après correction :
[np.float64(1.0), np.float64(10.0), np.float64(13.0), np.float64(21.0)]


## Choix : conservation des pays hors nomenclature

Plutôt que de supprimer les répondants dont le pays n'existe pas dans la base d'enquête — ce qui réduirait fortement un échantillon déjà petit —, on les conserve dans la base concaténée. Les analyses limitées à un périmètre géographique filtrent a posteriori sur `Country` (voir N6).

## Concaténation des deux bases

`pd.concat([...], ignore_index=True)` empile les lignes en alignant les colonnes de même nom et réinitialise l'index. Les colonnes présentes dans une seule source sont automatiquement complétées par `NaN` pour les lignes de l'autre source. Le bloc vérifie que le nombre de lignes obtenu est bien la somme des deux bases.

In [7]:
df_concat = pd.concat([df2, df_quest], ignore_index=True)

print(f"Base concaténée : {df_concat.shape[0]} lignes × {df_concat.shape[1]} colonnes")
print(f"\nVérification :")
print(f"  - Lignes de la base immigration : {df2.shape[0]}")
print(f"  - Lignes du questionnaire : {df_quest.shape[0]}")
print(f"  - Total attendu : {df2.shape[0] + df_quest.shape[0]}")
print(f"  - Total obtenu : {df_concat.shape[0]}")

Base concaténée : 5062 lignes × 44 colonnes

Vérification :
  - Lignes de la base immigration : 5000
  - Lignes du questionnaire : 62
  - Total attendu : 5062
  - Total obtenu : 5062


## Diagnostic des manquants après concaténation

Le taux de manquants par colonne fait apparaître la structure créée par la concaténation : les colonnes communes sont quasi complètes, tandis que les colonnes propres au questionnaire sont manquantes pour toutes les lignes issues de la base d'enquête. Ces colonnes très majoritairement vides constituent la difficulté méthodologique de l'étape suivante.

In [8]:
manquants = (df_concat.isna().sum() / len(df_concat) * 100).round(2)
manquants = manquants[manquants > 0].sort_values(ascending=False)

print("Pourcentage de manquants par colonne après concaténation :\n")
print(manquants)

Pourcentage de manquants par colonne après concaténation :

Taille_menage                   98.78
Revenu                          98.78
Decideur_financier              98.78
Source_revenu_salariat          98.78
Source_revenu_aide_familiale    98.78
Source_revenu_agriculture       98.78
Source_revenu_commerce          98.78
Source_revenu_transferts        98.78
Source_revenu_autre             98.78
Choc_economique                 98.78
Frequence_virements             98.78
Proches_europe                  98.78
Proche_fratrie                  98.78
Acces_credit                    98.78
Pays_etape                      98.78
Nationalite_francaise           98.78
Mariage_blanc                   98.78
Raisons_migration               98.78
Retour_pays                     98.78
Proche_autre                    98.78
Intention_envoyer_argent        98.78
Proche_parent                   98.78
Proche_ami                      98.78
Montant_envois                  98.78
Frequence_envois            

## Imputation par Monte Carlo

Pour chaque colonne incomplète, les valeurs manquantes sont remplacées par des tirages aléatoires avec remise (`np.random.choice`) parmi les valeurs observées. La distribution marginale de chaque variable est ainsi préservée.

**Limite majeure** : le tirage est indépendant des autres variables. Il ne tient pas compte du profil du répondant et annule donc mécaniquement toute corrélation entre une variable imputée et le reste de la base. Des méthodes conditionnelles (hot-deck par strates, MICE) permettraient de préserver ces relations.

`np.random.seed(42)` rend la simulation reproductible : deux exécutions produisent exactement les mêmes valeurs imputées.

In [9]:
np.random.seed(42)

colonnes_a_imputer = manquants.index.tolist()

print(f"Imputation Monte Carlo sur {len(colonnes_a_imputer)} colonnes\n")

for col in colonnes_a_imputer:
    valeurs_observees = df_concat[col].dropna().values

    if len(valeurs_observees) == 0:
        print(f"  {col} : aucune valeur observée, on saute")
        continue

    indices_manquants = df_concat[df_concat[col].isna()].index
    n_manquants = len(indices_manquants)

    valeurs_imputees = np.random.choice(valeurs_observees, size=n_manquants, replace=True)
    df_concat.loc[indices_manquants, col] = valeurs_imputees

    print(f"  {col} : {n_manquants} valeurs imputées (à partir de {len(valeurs_observees)} valeurs observées)")

Imputation Monte Carlo sur 27 colonnes

  Taille_menage : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Revenu : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Decideur_financier : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_salariat : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_aide_familiale : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_agriculture : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_commerce : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_transferts : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Source_revenu_autre : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Choc_economique : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Frequence_virements : 5000 valeurs imputées (à partir de 62 valeurs observées)
  Proches_europe : 5000 valeurs imputées (à partir de 62 valeurs ob

## Vérification finale

Le nombre total de manquants doit être nul. Le bloc suivant compare ensuite, pour quelques variables, la distribution observée dans le questionnaire et la distribution après imputation : elles doivent être proches si le tirage respecte bien les proportions d'origine.

In [10]:
manquants_final = df_concat.isna().sum()
manquants_final = manquants_final[manquants_final > 0]

if len(manquants_final) == 0:
    print("Aucun manquant — la base concaténée est complète")
else:
    print("Manquants restants :")
    print(manquants_final)

print(f"\nDimensions finales : {df_concat.shape[0]} lignes × {df_concat.shape[1]} colonnes")

Aucun manquant — la base concaténée est complète

Dimensions finales : 5062 lignes × 44 colonnes


In [11]:
print("Comparaison des distributions avant/après imputation\n")

for col in ["Revenu", "Choc_economique", "Nationalite_francaise", "Proches_europe", "Retour_pays"]:
    if col in df_concat.columns:
        print(f"--- {col} ---")
        print(f"Distribution observée ({len(df_quest)} répondants questionnaire) :")
        print(df_quest[col].value_counts(normalize=True).round(3).sort_index() * 100)
        print(f"\nDistribution après imputation ({len(df_concat)} lignes) :")
        print(df_concat[col].value_counts(normalize=True).round(3).sort_index() * 100)
        print()

Comparaison des distributions avant/après imputation

--- Revenu ---
Distribution observée (62 répondants questionnaire) :
Revenu
1    25.8
2     8.1
3     8.1
4    38.7
5    12.9
6     6.5
Name: proportion, dtype: float64

Distribution après imputation (5062 lignes) :
Revenu
1.0    26.7
2.0     8.3
3.0     7.3
4.0    37.9
5.0    12.7
6.0     7.0
Name: proportion, dtype: float64

--- Choc_economique ---
Distribution observée (62 répondants questionnaire) :
Choc_economique
0    88.7
1    11.3
Name: proportion, dtype: float64

Distribution après imputation (5062 lignes) :
Choc_economique
0.0    89.8
1.0    10.2
Name: proportion, dtype: float64

--- Nationalite_francaise ---
Distribution observée (62 répondants questionnaire) :
Nationalite_francaise
0    54.8
1    45.2
Name: proportion, dtype: float64

Distribution après imputation (5062 lignes) :
Nationalite_francaise
0.0    55.2
1.0    44.8
Name: proportion, dtype: float64

--- Proches_europe ---
Distribution observée (62 répondants que

## Export de la base concaténée

Export au format `.xlsx` ; ce fichier est contrôlé par N5 avant les analyses descriptives de N6.

In [12]:
df_concat.to_excel(
    "../data/3_Donnees_fusionnees/base_concatenee_finale.xlsx",
    index=False
)

print(f"Base concaténée exportée : base_concatenee_finale.xlsx")
print(f"Dimensions : {df_concat.shape[0]} lignes × {df_concat.shape[1]} colonnes")
print(f"Manquants restants : {df_concat.isna().sum().sum()}")

Base concaténée exportée : base_concatenee_finale.xlsx
Dimensions : 5062 lignes × 44 colonnes
Manquants restants : 0
